### dataclasses

In [ ]:
# --- Agent-level structured output via response_format (NOTES.md §6, §9) ---
# Same three schema styles (Pydantic here, TypedDict and dataclass in the next cells)
# work as an agent's response_format: the agent runs its loop, then returns a typed
# object in result["structured_response"]. Pydantic is the go-to for LLM output
# because it validates/coerces the model's text.
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from pydantic import BaseModel, Field

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name:str=Field(description="The name of the person")
    email:str=Field(description="The email of the person")
    phone:str=Field(description="The phone number of the person")

model=init_chat_model("ollama:qwen3:8b")
agent=create_agent(
    model=model,
    response_format=ContactInfo   # agent returns a validated ContactInfo
) 

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: Chris Nolan, chris@nolan.com, (+91) 9820897765"}]
})
print(result["structured_response"])

In [2]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from typing_extensions import TypedDict

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name:str
    email:str
    phone:str

model=init_chat_model("ollama:qwen3:8b")
agent=create_agent(
    model=model,
    response_format=ContactInfo   
) 

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: Chris Nolan, chris@nolan.com, (+91) 9820897765"}]
})
print(result["structured_response"])

{'name': 'Chris Nolan', 'email': 'chris@nolan.com', 'phone': '(+91) 9820897765'}


In [ ]:
## dataclass — lightweight, no dependency, real objects (attribute access, x.name).
## Best for internal/config data that does NOT cross a trust boundary; for untrusted
## LLM output Pydantic's validation is preferable (NOTES.md §9).
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name:str
    email:str
    phone:str

model=init_chat_model("ollama:qwen3:8b")
agent=create_agent(
    model=model,
    response_format=ContactInfo   
) 

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: Chris Nolan, chris@nolan.com, (+91) 9820897765"}]
})
print(result["structured_response"])